<a href="https://colab.research.google.com/github/achinthyam08/AI-LAB/blob/main/1BF24CS014_Week2_8_puzzle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from collections import deque

# Define the target configuration (1-8 in order, with 0 at the end)
GOAL_STATE = (1, 2, 3, 8, 0, 4, 7, 6, 5)

# 3x3 grid neighborhood mapping based on flat index string/tuple array
# Each index maps to valid adjacent flat indices it can swap with (Up, Down, Left, Right)
MOVES = {
    0: [3, 1],       # Can move down to 3, right to 1
    1: [4, 0, 2],    # Can move down to 4, left to 0, right to 2
    2: [5, 1],       # Can move down to 5, left to 1
    3: [0, 6, 4],    # Can move up to 0, down to 6, right to 4
    4: [1, 7, 3, 5], # Can move up to 1, down to 7, left to 3, right to 5
    5: [2, 8, 4],    # Can move up to 2, down to 8, left to 4
    6: [3, 7],       # Can move up to 3, right to 7
    7: [4, 6, 8],    # Can move up to 4, left to 6, right to 8
    8: [5, 7]        # Can move up to 5, left to 7
}

def get_neighbors(state):
    """Generates all possible next states from the current state."""
    neighbors = []
    zero_idx = state.index(0)
    for next_idx in MOVES[zero_idx]:
        # Swap the blank space with the neighboring tile
        new_state = list(state)
        new_state[zero_idx], new_state[next_idx] = new_state[next_idx], new_state[zero_idx]
        neighbors.append(tuple(new_state))
    return neighbors

# =====================================================================
# 1. BREADTH-FIRST SEARCH (BFS)
# =====================================================================
def solve_bfs(start_state):
    """Solves the 8-puzzle using a Queue structure (FIFO)."""
    queue = deque([(start_state, [])])
    visited = {start_state}

    while queue:
        current, path = queue.popleft()

        if current == GOAL_STATE:
            return path + [current]

        for neighbor in get_neighbors(current):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [current]))
    return None

# =====================================================================
# 2. DEPTH-FIRST SEARCH (DFS) - Iterative with Depth Limit
# =====================================================================
def solve_dfs(start_state, max_depth=30):
    """
    Solves the 8-puzzle using a Stack structure (LIFO).
    Note: Pure standard DFS can fall into infinite deep branches or cycles
    in infinite graph spaces, so a depth ceiling is added here.
    """
    stack = [(start_state, [], 0)]
    visited = {} # Stores state -> minimum depth reached to allow re-evaluation

    while stack:
        current, path, depth = stack.pop()

        if current == GOAL_STATE:
            return path + [current]

        if depth >= max_depth:
            continue

        if current in visited and visited[current] <= depth:
            continue

        visited[current] = depth

        for neighbor in get_neighbors(current):
            stack.append((neighbor, path + [current], depth + 1))
    return None

# =====================================================================
# 3. ITERATIVE DEEPENING SEARCH (IDS)
# =====================================================================
def depth_limited_search(state, depth, path, visited):
    """Helper recursive depth-limited search function for IDS."""
    if state == GOAL_STATE:
        return path + [state]
    if depth <= 0:
        return None

    visited.add(state)
    for neighbor in get_neighbors(state):
        if neighbor not in visited:
            result = depth_limited_search(neighbor, depth - 1, path + [state], visited)
            if result is not None:
                return result
    visited.remove(state) # Backtrack
    return None

def solve_ids(start_state, max_depth=50):
    """Iteratively deepens the search constraint until a limit is satisfied."""
    for depth in range(max_depth):
        visited = set()
        result = depth_limited_search(start_state, depth, [], visited)
        if result is not None:
            return result
    return None

# =====================================================================
# UTILITY HELPER FOR GRID LAYOUT
# =====================================================================
def print_grid(state):
    """Formats the flat state tuple into a readable 3x3 layout."""
    for i in range(0, 9, 3):
        print(f" {state[i]} {state[i+1]} {state[i+2]} ")
    print("-" * 11)

# =====================================================================
# EXECUTION PIPELINE
# =====================================================================
if __name__ == "__main__":
    # Solvable initial board state (requires ~5 steps to reach goal)
    initial_board = (2, 8, 3, 1, 6, 4, 7, 0, 5)

    print("--- STARTING INITIAL BOARD STATE ---")
    print_grid(initial_board)

    # 1. Run BFS
    print("\n[Executing Breadth-First Search...]")
    bfs_sol = solve_bfs(initial_board)
    print(f"BFS Solution Found! Total Steps: {len(bfs_sol) - 1 if bfs_sol else 'Failed'}")

    # 2. Run DFS
    print("\n[Executing Depth-First Search (Depth Limit=20)...]")
    dfs_sol = solve_dfs(initial_board, max_depth=20)
    print(f"DFS Solution Found! Total Steps: {len(dfs_sol) - 1 if dfs_sol else 'Failed'}")

    # 3. Run IDS
    print("\n[Executing Iterative Deepening Search...]")
    ids_sol = solve_ids(initial_board)
    print(f"IDS Solution Found! Total Steps: {len(ids_sol) - 1 if ids_sol else 'Failed'}")

    # Display step-by-step traversal using BFS result
    if bfs_sol:
        print("\n--- STEP-BY-STEP SOLUTION PATH (BFS) ---")
        for index, step in enumerate(bfs_sol):
            print(f"Step {index}:")
            print_grid(step)


--- STARTING INITIAL BOARD STATE ---
 2 8 3 
 1 6 4 
 7 0 5 
-----------

[Executing Breadth-First Search...]
BFS Solution Found! Total Steps: 5

[Executing Depth-First Search (Depth Limit=20)...]
DFS Solution Found! Total Steps: 15

[Executing Iterative Deepening Search...]
IDS Solution Found! Total Steps: 5

--- STEP-BY-STEP SOLUTION PATH (BFS) ---
Step 0:
 2 8 3 
 1 6 4 
 7 0 5 
-----------
Step 1:
 2 8 3 
 1 0 4 
 7 6 5 
-----------
Step 2:
 2 0 3 
 1 8 4 
 7 6 5 
-----------
Step 3:
 0 2 3 
 1 8 4 
 7 6 5 
-----------
Step 4:
 1 2 3 
 0 8 4 
 7 6 5 
-----------
Step 5:
 1 2 3 
 8 0 4 
 7 6 5 
-----------


In [4]:
def print_board(board):
    """Displays the 3x3 puzzle board."""
    for i in range(0, 9, 3):
        print(board[i : i + 3])
    print("-" * 9)


def get_successors(state):
    """Generates valid next states by moving the blank tile (0)."""
    successors = []
    blank_idx = state.index(0)
    row, col = divmod(blank_idx, 3)

    # Possible movements: (row_change, col_change, move_name)
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for dr, dc, move in moves:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_blank_idx = new_row * 3 + new_col
            # Swap blank with the adjacent tile
            new_state = list(state)
            new_state[blank_idx], new_state[new_blank_idx] = (
                new_state[new_blank_idx],
                new_state[blank_idx],
            )
            successors.append((tuple(new_state), move))

    return successors


def dls(state, goal, depth, path):
    """Depth-Limited Search helper for IDS."""
    if state == goal:
        return path

    if depth <= 0:
        return None

    for next_state, move in get_successors(state):
        if next_state not in path:  # Prevent immediate loops in the current path
            result = dls(next_state, goal, depth - 1, path + [(next_state, move)])
            if result is not None:
                return result

    return None


def iterative_deepening_search(start, goal, max_depth=30):
    """Iterative Deepening Search (IDS) main function."""
    for depth in range(max_depth + 1):
        print(f"--- Searching at Depth Limit: {depth} ---")
        result = dls(start, goal, depth, [(start, "Start")])
        if result is not None:
            return result
    return None


# Example Usage:
# 0 represents the blank space
initial_state = (2, 8, 3, 1, 6, 4, 7, 0, 5)
goal_state = (1, 2, 3, 8, 0, 4, 7, 6, 5)

print("Initial State:")
print_board(initial_state)

solution_path = iterative_deepening_search(initial_state, goal_state, max_depth=15)

if solution_path:
    print(f"\nSolution found in {len(solution_path) - 1} moves! Intermediate states:")
    for idx, (st, move) in enumerate(solution_path):
        print(f"Step {idx}: Move -> {move}")
        print_board(st)
else:
    print("No solution found within the depth limit.")


Initial State:
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)
---------
--- Searching at Depth Limit: 0 ---
--- Searching at Depth Limit: 1 ---
--- Searching at Depth Limit: 2 ---
--- Searching at Depth Limit: 3 ---
--- Searching at Depth Limit: 4 ---
--- Searching at Depth Limit: 5 ---

Solution found in 5 moves! Intermediate states:
Step 0: Move -> Start
(2, 8, 3)
(1, 6, 4)
(7, 0, 5)
---------
Step 1: Move -> Up
(2, 8, 3)
(1, 0, 4)
(7, 6, 5)
---------
Step 2: Move -> Up
(2, 0, 3)
(1, 8, 4)
(7, 6, 5)
---------
Step 3: Move -> Left
(0, 2, 3)
(1, 8, 4)
(7, 6, 5)
---------
Step 4: Move -> Down
(1, 2, 3)
(0, 8, 4)
(7, 6, 5)
---------
Step 5: Move -> Right
(1, 2, 3)
(8, 0, 4)
(7, 6, 5)
---------


In [6]:
class PuzzleState:
    def __init__(self, board, parent=None, move="", depth=0):
        self.board = board       # 1D tuple representing the 3x3 grid
        self.parent = parent     # Pointer to the previous state
        self.move = move         # The move that led to this state
        self.depth = depth       # Current depth level in the search tree

    def __eq__(self, other):
        return self.board == other.board

    def __hash__(self):
        return hash(self.board)

def print_board(state_tuple):
    """Prints the 1D state representation into a 3x3 grid format."""
    for i in range(0, 9, 3):
        print(f" {state_tuple[i]} {state_tuple[i+1]} {state_tuple[i+2]} ")
    print("-" * 11)

def get_neighbors(state):
    """Generates all valid moves (Up, Down, Left, Right) from the current state."""
    neighbors = []
    board = list(state.board)
    blank_idx = board.index(0)  # Find the empty space '0'
    row, col = blank_idx // 3, blank_idx % 3

    # Define valid movements: (row_offset, col_offset, move_name)
    moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

    for dr, dc, move_name in moves:
        new_row, new_col = row + dr, col + dc
        if 0 <= new_row < 3 and 0 <= new_col < 3:
            new_idx = new_row * 3 + new_col
            # Copy and swap elements
            new_board = board[:]
            new_board[blank_idx], new_board[new_idx] = new_board[new_idx], new_board[blank_idx]

            # Create a child node
            child_state = PuzzleState(tuple(new_board), parent=state, move=move_name, depth=state.depth + 1)
            neighbors.append(child_state)

    return neighbors

def solve_8_puzzle_dfs(start_board, goal_board, max_depth=30):
    """Solves the 8-puzzle using Depth-Limited Depth-First Search."""
    start_node = PuzzleState(tuple(start_board))
    goal_tuple = tuple(goal_board)

    # Use a LIFO queue (stack) for DFS
    stack = [start_node]
    visited = set()

    while stack:
        current_node = stack.pop()

        # Goal test
        if current_node.board == goal_tuple:
            return current_node

        if current_node not in visited and current_node.depth <= max_depth:
            visited.add(current_node)

            # Get neighbors and push them to the stack
            for neighbor in get_neighbors(current_node):
                if neighbor not in visited:
                    stack.append(neighbor)

    return None

def print_solution_steps(solution_node):
    """Backtracks from the goal node to reconstruct and display all intermediate states."""
    path = []
    current = solution_node
    while current:
        path.append(current)
        current = current.parent
    path.reverse()  # Reverse to start from the initial state

    print(f"\n🎉 Solution found in {len(path) - 1} steps!")
    for idx, state in enumerate(path):
        if idx == 0:
            print(f"\n[Step {idx}] Initial State:")
        else:
            print(f"\n[Step {idx}] Move blank tile: {state.move}")
        print_board(state.board)

# --- Execution ---
if __name__ == "__main__":
    # Representing a 3x3 grid as a flat 1D array. '0' represents the blank space.
    # Note: Keep the initial configuration close to the goal configuration
    # because DFS is inherently slow and non-optimal for short paths.
    initial_state = [2, 8, 3,
                     1, 6, 4,
                     7, 0, 5]

    goal_state    = [1, 2, 3,
                     8, 0, 4,
                     7, 6, 5]

    print("Starting DFS Search...")
    solution = solve_8_puzzle_dfs(initial_state, goal_state, max_depth=20)

    if solution:
        print_solution_steps(solution)
    else:
        print("\n❌ No solution found within the specified depth limit.")


Starting DFS Search...

🎉 Solution found in 19 steps!

[Step 0] Initial State:
 2 8 3 
 1 6 4 
 7 0 5 
-----------

[Step 1] Move blank tile: Right
 2 8 3 
 1 6 4 
 7 5 0 
-----------

[Step 2] Move blank tile: Up
 2 8 3 
 1 6 0 
 7 5 4 
-----------

[Step 3] Move blank tile: Left
 2 8 3 
 1 0 6 
 7 5 4 
-----------

[Step 4] Move blank tile: Left
 2 8 3 
 0 1 6 
 7 5 4 
-----------

[Step 5] Move blank tile: Up
 0 8 3 
 2 1 6 
 7 5 4 
-----------

[Step 6] Move blank tile: Right
 8 0 3 
 2 1 6 
 7 5 4 
-----------

[Step 7] Move blank tile: Down
 8 1 3 
 2 0 6 
 7 5 4 
-----------

[Step 8] Move blank tile: Right
 8 1 3 
 2 6 0 
 7 5 4 
-----------

[Step 9] Move blank tile: Down
 8 1 3 
 2 6 4 
 7 5 0 
-----------

[Step 10] Move blank tile: Left
 8 1 3 
 2 6 4 
 7 0 5 
-----------

[Step 11] Move blank tile: Up
 8 1 3 
 2 0 4 
 7 6 5 
-----------

[Step 12] Move blank tile: Up
 8 0 3 
 2 1 4 
 7 6 5 
-----------

[Step 13] Move blank tile: Left
 0 8 3 
 2 1 4 
 7 6 5 
-----------

[